# 21 Agent 与 agent loop（Agents & the Agent Loop） / Agents & the Agent Loop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BaiYuBY/use_token_to_learn/blob/main/docs/21-agents/21-agents.ipynb)

[📖 讲解 / Notes](https://baiyuby.github.io/use_token_to_learn/21-agents/README.html) · [🎮 交互演示 / Playground](https://baiyuby.github.io/use_token_to_learn/21-agents/interactive/agents.html) · [🧑‍💻 GitHub](https://github.com/BaiYuBY/use_token_to_learn/tree/main/docs/21-agents)

这个 notebook 由 `scripts/make_notebooks.py` 根据本章 `code/` 目录下的脚本自动生成，代码与脚本完全一致：每个函数或类是一个单元格，
可以逐个修改、重新运行。最后一个单元格运行本章的演示（带 assert 检查）。

Generated by `scripts/make_notebooks.py` from this chapter's `code/` scripts, with identical code: one cell per function or
class, so you can edit and re-run them one at a time. The last cell runs the chapter's demo (with its assert checks).

In [ ]:
# 环境 / setup: find the repository, or clone it (e.g. in Colab); some chapters import code from earlier ones
# 找到仓库，找不到就克隆一份（例如在 Colab 中）；有些章节会导入前面章节的代码
import subprocess
import sys
from pathlib import Path

CHAPTER = "21-agents"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs" / CHAPTER).is_dir()), None)
if ROOT is None:
    ROOT = Path.cwd() / "use_token_to_learn"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/BaiYuBY/use_token_to_learn.git", str(ROOT)], check=True)
CODE = ROOT / "docs" / CHAPTER / "code"
sys.path.insert(0, str(CODE))
print("code:", CODE)
try:
    import langgraph  # noqa: F401
except ImportError:  # e.g. in Colab / 例如在 Colab 中
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.19"], check=True)

## `agent_loop.py`

```text
The agent loop, written by hand / 手写 agent loop.

Run / 运行:
    python docs/21-agents/code/agent_loop.py

Covers / 包含:
    1. tools described by JSON schemas, and validating the model's tool calls against them
       用 JSON schema 描述工具，并按 schema 校验模型发出的工具调用
    2. the loop: call the model, run the tools it asks for, append the results, repeat until it answers
       循环：调用模型，执行它要求的工具，把结果追加进对话，直到它给出答案
    3. why long tasks fail: a small per-step error rate compounds; which errors the loop can recover from
       长任务为什么失败：每步很小的错误率会累积；循环能从哪些错误中恢复
    4. a self-check before answering, a step budget, the growing context, and parallel tool calls
       作答前的自检、步数上限、不断增长的上下文，以及并行工具调用

The "model" here is a script that follows a ReAct-style plan and makes mistakes at a set rate, so the
loop itself is easy to see and measure; chapter 18's tiny models are too small to plan tool use.
这里的“模型”是一个按 ReAct 方式规划、并以设定的概率犯错的脚本，这样循环本身就清清楚楚、可以测量；18 章那种小模型还不会规划工具调用。
```

In [ ]:
__file__ = str(CODE / "agent_loop.py")  # the code below finds its data files from here / 代码据此定位数据文件

In [ ]:
import json
import random
import re
from collections import defaultdict

### A small world and its tools / 小世界与工具

In [ ]:
SYL_A = "ma ve li to sa ri ka no be da lu fe mi ro ta ne si ko ga pe".split()
SYL_B = "ra len vik dor min sel tar nis bek lov run sha mar kel dun fis gal ton rik per".split()
YEAR = 2026

In [ ]:
def make_world(n_people=200, n_companies=20, seed=0):
    rng = random.Random(seed)
    used = set()

    def word():
        while True:
            w = (rng.choice(SYL_A) + rng.choice(SYL_B)).capitalize()
            if w not in used:
                used.add(w)
                return w

    cities = [word() + rng.choice(["vik", "berg", "ton"]) for _ in range(10)]
    companies = {f"{word()} {rng.choice(['Bank', 'Air', 'Labs', 'Works'])}": dict(city=rng.choice(cities), founded=rng.randint(1900, 2015))
                 for _ in range(n_companies)}
    people = {}
    while len(people) < n_people:  # first and last names may repeat, full names may not / 名和姓可以重复，全名不行
        first, last = (rng.choice(SYL_A) + rng.choice(SYL_B)).capitalize(), (rng.choice(SYL_A) * 2 + rng.choice(SYL_B)).capitalize()
        name = f"{first} {last}"
        if name in people or first in used or last in used:
            continue
        people[name] = dict(works=rng.choice(list(companies)), born=rng.choice(cities))
    docs = {p: f"{p} was born in {a['born']}. {p} works at {a['works']}." for p, a in people.items()}
    docs |= {c: f"{c} was founded in {a['founded']}. Its headquarters are in {a['city']}." for c, a in companies.items()}
    return dict(people=people, companies=companies, docs=docs)

In [ ]:
TOOLS = [  # what the model is told about each tool / 告诉模型的工具说明
    {"name": "search", "description": "Search the documents. Returns the titles of the best matches.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
    {"name": "read", "description": "Read the document with this exact title.",
     "parameters": {"type": "object", "properties": {"title": {"type": "string"}}, "required": ["title"]}},
    {"name": "calculator", "description": "Evaluate an arithmetic expression such as 2026 - 1950.",
     "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}},
    {"name": "final_answer", "description": "Give the final answer to the user and stop.",
     "parameters": {"type": "object", "properties": {"answer": {"type": "string"}}, "required": ["answer"]}},
]
SCHEMA = {t["name"]: t["parameters"] for t in TOOLS}

In [ ]:
def validate(call):
    """Check a tool call against its JSON schema; return an error message or None.
    按 JSON schema 检查工具调用；返回错误信息或 None。"""
    if not isinstance(call, dict) or call.get("name") not in SCHEMA:
        return f"unknown tool {call.get('name') if isinstance(call, dict) else call!r}"
    spec, args = SCHEMA[call["name"]], call.get("args")
    if not isinstance(args, dict):
        return "args must be a JSON object"
    missing = [k for k in spec["required"] if k not in args]
    wrong = [k for k, v in args.items() if k in spec["properties"] and not isinstance(v, str)]
    return f"missing {missing}" if missing else f"{wrong} must be strings" if wrong else None

In [ ]:
def run_tool(world, name, args):
    docs = world["docs"]
    if name == "search":
        q = set(re.findall(r"\w+", args["query"].lower()))
        scored = sorted(docs, key=lambda t: -len(q & set(re.findall(r"\w+", (t + " " + docs[t]).lower()))))
        return "Titles: " + "; ".join(scored[:3])
    if name == "read":
        return docs.get(args["title"], f"No document titled {args['title']!r}.")
    if name == "calculator":
        if not re.fullmatch(r"[\d\s+\-*/().]+", args["expression"]):
            return "Error: only arithmetic is allowed."
        return str(eval(args["expression"]))  # digits and operators only / 只含数字和运算符
    raise ValueError(name)

### Tasks / 任务

In [ ]:
def make_tasks(world, n=200, seed=1):
    """Questions needing 1 to 5 tool calls before the answer, with the true answer.
    需要 1 到 5 次工具调用的问题，以及正确答案。"""
    rng = random.Random(seed)
    people, companies = world["people"], world["companies"]
    tasks = defaultdict(list)
    for _ in range(n):
        p, q = rng.sample(list(people), 2)
        c, d = people[p]["works"], people[q]["works"]
        tasks[1].append((f"Where does {p} work?", c))
        tasks[2].append((f"In which city is the headquarters of the company {p} works for?", companies[c]["city"]))
        tasks[3].append((f"How many years ago (in {YEAR}) was the company {p} works for founded?", str(YEAR - companies[c]["founded"])))
        tasks[5].append((f"How many years apart were the companies of {p} and {q} founded?", str(abs(companies[c]["founded"] - companies[d]["founded"]))))
    return tasks

### The "model": a scripted ReAct policy with injected mistakes / “模型”：带注入错误的脚本化 ReAct 策略

In [ ]:
class ScriptedModel:
    """Reads the conversation so far and returns the next tool call(s), like a chat model with tool use.
    With probability error_rate per call it makes one of three mistakes:
      malformed  - a call that breaks the schema (recoverable: the loop returns the validation error)
      wrong_arg  - a misspelled title (recoverable: the tool says "no document")
      hallucinate- answers straight away with a plausible but made-up value (not recoverable,
                   unless something checks the answer)
    读取到目前为止的对话，返回下一个（或几个）工具调用，就像支持工具调用的对话模型。每次调用以 error_rate 的概率犯三种错之一：
    格式错误（可恢复）、参数拼错（可恢复）、直接编一个答案（不可恢复，除非有东西检查答案）。"""

    def __init__(self, world, error_rate=0.0, mix=(0.35, 0.35, 0.3), parallel=False, seed=0):
        self.world, self.error_rate, self.mix, self.parallel = world, error_rate, mix, parallel
        self.rng = random.Random(seed)
        self.calls = 0

    def __call__(self, messages):
        self.calls += 1
        question = messages[1]["content"]
        observed = " ".join(m["content"] for m in messages if m["role"] == "tool")
        plan = self._plan(question, observed)
        if self.rng.random() < self.error_rate:
            kind = self.rng.choices(["malformed", "wrong_arg", "hallucinate"], self.mix)[0]
            if kind == "wrong_arg" and plan[0]["name"] != "read":
                kind = "malformed"  # nothing to misspell; keep the share of fatal mistakes fixed / 没有可拼错的参数
            if kind == "malformed":
                return [{"name": plan[0]["name"], "args": json.dumps(plan[0]["args"])}]  # args as a string / 参数写成了字符串
            if kind == "wrong_arg":
                return [{"name": "read", "args": {"title": plan[0]["args"]["title"].lower()}}]
            if kind == "hallucinate":
                fake = self.rng.choice(list(self.world["companies"]) + [str(self.rng.randint(1, 120))])
                return [{"name": "final_answer", "args": {"answer": fake}}]
        return plan if self.parallel else plan[:1]

    def _plan(self, question, observed):
        """What a competent model would do next, given what it has seen. 一个合格的模型下一步会做什么。"""
        world = self.world
        names = re.findall(r"(?:of|does|company) ([A-Z]\w+ [A-Z]\w+)", question)
        names += [n for n in re.findall(r"and ([A-Z]\w+ [A-Z]\w+) founded", question) if n not in names]
        employers = [re.search(rf"{re.escape(n)} works at ([A-Z]\w+ \w+)\.", observed) for n in names]
        unread = [n for n, e in zip(names, employers) if not e]
        if unread:
            return [{"name": "read", "args": {"title": n}} for n in unread]
        companies = [e.group(1) for e in employers]
        if question.startswith("Where does"):
            return [{"name": "final_answer", "args": {"answer": companies[0]}}]
        facts = [re.search(rf"{re.escape(c)} was founded in (\d+)\. Its headquarters are in (\w+)\.", observed) for c in companies]
        missing = [c for c, f in zip(companies, facts) if not f]
        if missing:
            return [{"name": "read", "args": {"title": c}} for c in dict.fromkeys(missing)]
        if "headquarters" in question:
            return [{"name": "final_answer", "args": {"answer": facts[0].group(2)}}]
        expr = f"{YEAR} - {facts[0].group(1)}" if "years ago" in question else \
            f"{max(int(f.group(1)) for f in facts)} - {min(int(f.group(1)) for f in facts)}"
        result = re.search(rf"{re.escape(expr)} = (\d+)", observed)
        if not result:
            return [{"name": "calculator", "args": {"expression": expr}}]
        return [{"name": "final_answer", "args": {"answer": result.group(1)}}]

### The loop / 循环

In [ ]:
def supported(answer, messages):
    """Self-check: is the answer literally present in some tool result? 自检：答案是否真的出现在某个工具结果里。"""
    return any(m["role"] == "tool" and answer in m["content"] for m in messages)

In [ ]:
def agent_loop(model, world, question, max_steps=10, check=False, summarize=False):
    """The whole agent: ask the model, run the tools it calls, feed the results back, stop at
    final_answer or after max_steps model calls. Returns the answer (or None), the steps used, the
    words the model had to read in total, and the transcript.
    整个 agent：问模型、执行它调用的工具、把结果喂回去，遇到 final_answer 或模型调用达到 max_steps 次时停止。"""
    messages = [{"role": "system", "content": "You answer questions using the tools. " + json.dumps(TOOLS)},
                {"role": "user", "content": question}]
    read_words = 0
    for step in range(1, max_steps + 1):
        context = messages if not summarize else messages[:2] + [m for m in messages[2:] if m["role"] == "tool" and not m["content"].startswith(("Titles", "Error", "No document", "Check"))]
        read_words += sum(len(m["content"].split()) for m in context)
        calls = model(context)
        messages.append({"role": "assistant", "content": json.dumps(calls)})
        for call in calls:
            err = validate(call)
            if err:
                messages.append({"role": "tool", "content": f"Error: {err}. Call the tool again with valid arguments."})
                continue
            if call["name"] == "final_answer":
                answer = call["args"]["answer"]
                if check and not supported(answer, messages):
                    messages.append({"role": "tool", "content": "Check failed: that answer is not in any tool result. Keep working."})
                    continue
                return answer, step, read_words, messages
            out = run_tool(world, call["name"], call["args"])
            if call["name"] == "calculator":
                out = f"{call['args']['expression']} = {out}"
            messages.append({"role": "tool", "content": out})
    return None, max_steps, read_words, messages

In [ ]:
def evaluate(world, tasks, error_rate, check=False, max_steps=10, parallel=False, summarize=False, seed=0):
    model = ScriptedModel(world, error_rate, parallel=parallel, seed=seed)
    ok = steps = words = 0
    for question, truth in tasks:
        answer, n, w, _ = agent_loop(model, world, question, max_steps, check, summarize)
        ok += answer == truth
        steps += n
        words += w
    n = len(tasks)
    return ok / n, steps / n, words / n

In [ ]:
def main():
    print("== 1. tools as JSON schemas / 用 JSON schema 描述工具 ==")
    world = make_world()
    print("  " + json.dumps(TOOLS[1]))
    assert validate({"name": "read", "args": {"title": "x"}}) is None
    assert validate({"name": "read", "args": '{"title": "x"}'}) == "args must be a JSON object"
    assert validate({"name": "delete_all", "args": {}}).startswith("unknown tool")

    print("\n== 2. the loop / 循环 ==")
    tasks = make_tasks(world)
    question, truth = tasks[5][0]
    answer, steps, _, messages = agent_loop(ScriptedModel(world), world, question)
    for m in messages[1:]:
        print(f"  {m['role']:9s} {m['content'][:110]}")
    assert answer == truth and steps <= 6
    for k, ts in tasks.items():
        assert evaluate(world, ts, 0.0)[0] == 1.0  # a perfect model solves everything / 完美的模型全部答对

    print("\n== 3. errors compound / 错误会累积 ==")
    table = {}
    for e in [0.05, 0.1, 0.2]:
        row = {k: evaluate(world, ts, e) for k, ts in tasks.items()}
        table[e] = row
        print(f"  error rate {e:.0%} per model call: " + ", ".join(f"{k} tool call(s) {v[0]:.0%}" for k, v in row.items()))
    assert table[0.2][1][0] > table[0.2][5][0] + 0.1      # longer tasks fail more / 越长越容易失败
    print("  mistakes the loop recovers from (malformed calls, wrong titles) cost steps; made-up answers end the run")

    print("\n== 4. a self-check, the step budget, the context, parallel calls / 自检、步数上限、上下文、并行 ==")
    plain, checked = evaluate(world, tasks[5], 0.1), evaluate(world, tasks[5], 0.1, check=True)
    print(f"  5-call tasks at 10% errors: {plain[0]:.0%} -> {checked[0]:.0%} with a self-check before answering "
          f"({plain[1]:.1f} -> {checked[1]:.1f} model calls)")
    assert checked[0] > plain[0] + 0.05
    for budget in [6, 8, 12]:
        print(f"  max {budget:2d} model calls: {evaluate(world, tasks[5], 0.1, check=True, max_steps=budget)[0]:.0%}")
    full, short = evaluate(world, tasks[5], 0.0), evaluate(world, tasks[5], 0.0, summarize=True)
    print(f"  words the model reads per task: {full[2]:.0f} with the full transcript, {short[2]:.0f} keeping only the useful results")
    assert short[2] < full[2]
    serial, parallel = evaluate(world, tasks[5], 0.0), evaluate(world, tasks[5], 0.0, parallel=True)
    print(f"  model calls per 5-call task: {serial[1]:.1f} one tool at a time, {parallel[1]:.1f} with parallel tool calls")
    assert parallel[1] < serial[1]

### 运行演示 / Run the demo (`agent_loop.py`)

In [ ]:
main()

## `langgraph_agent.py`

```text
The same agent with LangChain and LangGraph / 用 LangChain 和 LangGraph 实现同一个 agent.

Run / 运行 (needs `pip install langgraph`, which brings langchain-core):
    python docs/21-agents/code/langgraph_agent.py

Covers / 包含:
    1. LangChain tools: a typed Python function with a docstring becomes a JSON-schema tool
       LangChain 工具：带类型和文档字符串的 Python 函数自动变成 JSON schema 工具
    2. the agent loop as a LangGraph StateGraph: an "agent" node, a "tools" node, a conditional edge,
       plus a "check" node; it takes exactly the same steps as the hand-written loop
       把 agent loop 写成 LangGraph 的状态图：agent 节点、tools 节点、条件边，再加一个 check 节点；
       它走的每一步都和手写循环相同
    3. checkpoints: every step's state is saved and can be inspected or resumed
       检查点：每一步的状态都被保存，可以查看、可以从中断处继续
    4. human in the loop: pause before a risky tool and wait for approval
       人工介入：执行有风险的工具前暂停，等人批准
    5. LCEL: composing runnables with |, as a retrieval chain from chapter 19
       LCEL：用 | 组合可运行对象，例如 19 章的检索链
```

In [ ]:
__file__ = str(CODE / "langgraph_agent.py")  # the code below finds its data files from here / 代码据此定位数据文件

In [ ]:
import json
import sys
from pathlib import Path

In [ ]:
sys.path.insert(0, str(Path(__file__).resolve().parent))

In [ ]:
from agent_loop import ScriptedModel, agent_loop, make_tasks, make_world, run_tool, supported  # noqa: E402

In [ ]:
try:
    from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
    from langchain_core.runnables import RunnableLambda
    from langchain_core.tools import tool
    from langchain_core.utils.function_calling import convert_to_openai_tool
    from langgraph.checkpoint.memory import InMemorySaver
    from langgraph.graph import END, START, MessagesState, StateGraph
    from langgraph.prebuilt import ToolNode
    from langgraph.types import Command, interrupt
except ImportError:
    raise SystemExit("This file needs LangGraph: pip install langgraph==1.2.19 / 需要先安装 LangGraph")
WORLD = make_world()

### 1. Tools / 工具

In [ ]:
@tool
def search(query: str) -> str:
    """Search the documents. Returns the titles of the best matches."""
    return run_tool(WORLD, "search", {"query": query})

In [ ]:
@tool
def read(title: str) -> str:
    """Read the document with this exact title."""
    return run_tool(WORLD, "read", {"title": title})

In [ ]:
@tool
def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression such as 2026 - 1950."""
    return f"{expression} = {run_tool(WORLD, 'calculator', {'expression': expression})}"

In [ ]:
TOOLS = [search, read, calculator]

### 2. The agent as a state graph / 状态图形式的 agent

In [ ]:
def to_plain(messages):
    """LangChain messages -> the plain dicts the scripted model reads. LangChain 消息 -> 脚本模型读取的字典。"""
    out = [{"role": "system", "content": ""}]
    for m in messages:
        if isinstance(m, HumanMessage):
            out.append({"role": "user", "content": m.content})
        elif isinstance(m, ToolMessage):
            out.append({"role": "tool", "content": m.content})
        else:
            out.append({"role": "assistant", "content": m.content})
    return out

In [ ]:
def build_agent(model, check=True, checkpointer=None):
    """START -> agent -> (tool calls? -> tools -> agent) -> (final answer -> check -> END or back to agent).
    The prebuilt ToolNode runs the tools; the routing functions are the loop's if-statements.
    预置的 ToolNode 负责执行工具；路由函数就是循环里的 if 判断。"""
    counter = {"n": 0}

    def agent(state: MessagesState):
        calls = model(to_plain(state["messages"]))
        tool_calls = [c for c in calls if c["name"] != "final_answer"]
        counter["n"] += 1
        if tool_calls:
            # a malformed call (arguments not an object) goes through with no arguments; ToolNode then
            # returns the validation error to the model, as the hand-written loop does
            # 格式错误的调用（参数不是对象）以空参数传过去；ToolNode 会把校验错误返回给模型，和手写循环一样
            return {"messages": [AIMessage(content="", tool_calls=[
                {"name": c["name"], "args": c["args"] if isinstance(c["args"], dict) else {}, "id": f"call_{counter['n']}_{i}"}
                for i, c in enumerate(tool_calls)])]}
        return {"messages": [AIMessage(content=calls[0]["args"]["answer"])]}

    def route_after_agent(state: MessagesState):
        last = state["messages"][-1]
        return "tools" if last.tool_calls else ("check" if check else END)

    def check_answer(state: MessagesState):
        answer = state["messages"][-1].content
        if supported(answer, to_plain(state["messages"][:-1])):
            return {}
        return {"messages": [HumanMessage("Check failed: that answer is not in any tool result. Keep working.")]}

    def route_after_check(state: MessagesState):
        return "agent" if isinstance(state["messages"][-1], HumanMessage) else END

    graph = StateGraph(MessagesState)
    graph.add_node("agent", agent)
    graph.add_node("tools", ToolNode(TOOLS))
    graph.add_node("check", check_answer)
    graph.add_edge(START, "agent")
    graph.add_conditional_edges("agent", route_after_agent, ["tools", "check", END])
    graph.add_edge("tools", "agent")
    graph.add_conditional_edges("check", route_after_check, ["agent", END])
    return graph.compile(checkpointer=checkpointer)

In [ ]:
def tool_trace(messages):
    """The sequence of tool calls, for comparing two runs. 工具调用序列，用来比较两次运行。"""
    out = []
    for m in messages:
        if isinstance(m, AIMessage):
            out += [(c["name"], tuple(sorted(c["args"].items()))) for c in m.tool_calls]
        elif isinstance(m, dict) and m["role"] == "assistant":
            out += [(c["name"], tuple(sorted(c["args"].items()))) for c in json.loads(m["content"]) if c["name"] != "final_answer"]
    return out

### 4. Human in the loop / 人工介入

In [ ]:
def build_payment_agent(checkpointer):
    """A two-node graph: the agent wants to send money; the guard node pauses with interrupt() and waits
    for a person's decision, which arrives later as Command(resume=...).
    两个节点的图：agent 想转账；guard 节点用 interrupt() 暂停，等人的决定，之后通过 Command(resume=...) 传回来。"""

    def agent(state: MessagesState):
        last = state["messages"][-1]
        if isinstance(last, ToolMessage):
            return {"messages": [AIMessage(content=f"Done: {last.content}")]}
        return {"messages": [AIMessage(content="", tool_calls=[{"name": "transfer", "args": {"to": "Gasel Bank", "amount": 900}, "id": "t1"}])]}

    def guard(state: MessagesState):
        call = state["messages"][-1].tool_calls[0]
        decision = interrupt({"tool": call["name"], "args": call["args"], "question": "approve?"})
        result = f"sent {call['args']['amount']} to {call['args']['to']}" if decision == "yes" else "the user declined the transfer"
        return {"messages": [ToolMessage(content=result, tool_call_id=call["id"])]}

    graph = StateGraph(MessagesState)
    graph.add_node("agent", agent)
    graph.add_node("guard", guard)
    graph.add_edge(START, "agent")
    graph.add_conditional_edges("agent", lambda s: "guard" if s["messages"][-1].tool_calls else END, ["guard", END])
    graph.add_edge("guard", "agent")
    return graph.compile(checkpointer=checkpointer)

In [ ]:
def main():
    print("== 1. LangChain tools / LangChain 工具 ==")
    schema = convert_to_openai_tool(read)
    print(f"  @tool read(title: str) -> {schema['function']}")
    assert schema["function"]["parameters"]["required"] == ["title"]

    print("\n== 2. the same loop as a LangGraph state graph / 同一个循环，写成 LangGraph 状态图 ==")
    app = build_agent(ScriptedModel(WORLD))
    print("  " + app.get_graph().draw_mermaid().split("graph TD;")[1].strip().replace("\n", "\n  ")[:600])
    tasks = make_tasks(WORLD, n=30)
    same = 0
    total = 0
    for k, ts in tasks.items():
        for question, truth in ts:
            out = app.invoke({"messages": [HumanMessage(question)]})
            hand = agent_loop(ScriptedModel(WORLD), WORLD, question)
            same += out["messages"][-1].content == truth == hand[0] and tool_trace(out["messages"]) == tool_trace(hand[3])
            total += 1
    print(f"  {same}/{total} tasks: same answer and same tool calls as the hand-written loop")
    assert same == total

    print("\n== 3. checkpoints / 检查点 ==")
    saver = InMemorySaver()
    app = build_agent(ScriptedModel(WORLD, error_rate=0.3, seed=3), checkpointer=saver)
    config = {"configurable": {"thread_id": "demo"}}
    question, truth = tasks[5][0]
    out = app.invoke({"messages": [HumanMessage(question)]}, config)
    history = list(app.get_state_history(config))
    print(f"  {len(history)} checkpoints saved for one run; the latest has {len(history[0].values['messages'])} messages; "
          f"answer {out['messages'][-1].content!r} (truth {truth!r})")
    for snap in reversed(history[:-1]):
        if snap.next:
            print(f"    step {snap.metadata['step']:2d}: next node {snap.next[0]:6s} | messages so far {len(snap.values['messages'])}")
    assert len(history) > 5

    print("\n== 4. human in the loop / 人工介入 ==")
    for decision in ["yes", "no"]:
        app = build_payment_agent(InMemorySaver())
        config = {"configurable": {"thread_id": decision}}
        paused = app.invoke({"messages": [HumanMessage("Pay the invoice from Gasel Bank.")]}, config)
        request = paused["__interrupt__"][0].value
        out = app.invoke(Command(resume=decision), config)
        print(f"  paused before {request['tool']}({request['args']}); the person says {decision!r} -> {out['messages'][-1].content}")
        assert ("sent 900" in out["messages"][-1].content) == (decision == "yes")

    print("\n== 5. LCEL: a retrieval chain with | / LCEL：用 | 组成检索链 ==")
    retrieve = RunnableLambda(lambda q: search.invoke(q).removeprefix("Titles: ").split("; ")[0])
    fetch = RunnableLambda(lambda title: read.invoke(title))
    to_prompt = RunnableLambda(lambda doc: f"<source>{doc}</source>\nAnswer from the source.")
    chain = retrieve | fetch | to_prompt
    p = next(iter(WORLD["people"]))
    print("  " + chain.invoke(f"Where does {p} work?").replace("\n", "\n  "))
    print(f"  batch of 3 questions -> {len(chain.batch([f'Where does {q} work?' for q in list(WORLD['people'])[:3]]))} prompts")

### 运行演示 / Run the demo (`langgraph_agent.py`)

In [ ]:
main()

## 动手试试 / Try it yourself

改一改上面任意一个函数或参数，重新运行它所在的单元格，再运行 `main()` 看结果有什么变化。
assert 失败说明你的改动改变了程序检查的性质，这正是值得思考的地方。

Change any function or parameter above, re-run its cell, then run `main()` again. If an assert fails, your change broke a
property the script checks, which is exactly what is worth thinking about.